In [2]:
import torch.nn as nn
import torch
import torchvision
import torchvision.transforms.v2 as T
import numpy as np

toTensor = T.Compose([T.ToImage() , T.ToDtype(torch.float32 , scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(root="datasets" , download=True , train=True , transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(root="datasets" , download=True , train=False , transform=toTensor)

In [3]:
torch.manual_seed(42)
train_data , valid_data = torch.utils.data.random_split(train_and_valid_data , [55000 , 5000])

In [4]:
from torch.utils.data import DataLoader

train_data_loader = DataLoader(train_data , batch_size=32 , shuffle=True )
test_data_loader = DataLoader(test_data , batch_size = 32)
valid_data_loader = DataLoader(valid_data , batch_size= 32)

In [ ]:
#Classifier
class imageClassifier(nn.Module):
    def __init__(self, n_inputs , n_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs , 300),
            nn.ReLU(),
            nn.Linear(300, 200),
            nn.ReLU(),
            nn.Linear(200 , n_classes)
        )
    def forward(self , x):
        return self.model(x)
    
def train(model , criterion , optimizer , data_loader , n_epochs):
        model.to(device= "cuda")
        model.train()
        for epoch in range(n_epochs):
            total_loss = 0.0
            for x_batch, y_batch in data_loader:
                x_batch , y_batch = x_batch.to(device = "cuda") , y_batch.to(device = "cuda")
                y_pred = model(x_batch)
                loss = criterion(y_pred , y_batch)
                total_loss += loss.item()
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
            mean_loss = total_loss / len(data_loader)
            print(f"Epoch {epoch + 1} loss :{mean_loss}")

def evaluate(model , data_loader , metrics_fn  , aggregate_fn = lambda metric : torch.mean(metric)):
        model.to(device = "cuda")
        model.eval()
        metrics = []
        with torch.no_grad():
            for x_batch , y_batch in data_loader:
                x_batch , y_batch = x_batch.to(device = "cuda") , y_batch.to(device = "cuda")
                y_pred = model(x_batch)
                metric = metrics_fn(y_pred , y_batch)
                metrics.append(metric.detach())
            return aggregate_fn(torch.stack(metrics))

In [8]:
model= imageClassifier(n_inputs= 28 * 28 , n_classes=10)
optimizer = torch.optim.SGD(model.parameters() , lr= 0.01)
criterion = nn.CrossEntropyLoss()
n_epochs = 30

train(model , criterion , optimizer , train_data_loader , n_epochs)

Epoch1 loss :1.0943437282639648
Epoch2 loss :0.5901380331711134
Epoch3 loss :0.5067677744266043
Epoch4 loss :0.46818474433132906
Epoch5 loss :0.4440487916036842
Epoch6 loss :0.42374940934726985
Epoch7 loss :0.40934759124446013
Epoch8 loss :0.3955622446431262
Epoch9 loss :0.3835215515317856
Epoch10 loss :0.37244827262602137
Epoch11 loss :0.36300217503071736
Epoch12 loss :0.3535266261051532
Epoch13 loss :0.34547921128402
Epoch14 loss :0.3377191983835166
Epoch15 loss :0.32986510509907885
Epoch16 loss :0.3239588910879577
Epoch17 loss :0.3177219669769641
Epoch18 loss :0.31074985696426516
Epoch19 loss :0.3043790302212811
Epoch20 loss :0.3001394409538962
Epoch21 loss :0.29439637550586806
Epoch22 loss :0.28887226217582934
Epoch23 loss :0.2832454512260937
Epoch24 loss :0.278986321407879
Epoch25 loss :0.27552082763314073
Epoch26 loss :0.269299809379398
Epoch27 loss :0.26640774226850933
Epoch28 loss :0.2612562751838501
Epoch29 loss :0.25745564738614674
Epoch30 loss :0.2535478962696937


In [11]:
import torchmetrics

accuracy = torchmetrics.Accuracy(task="multiclass" , num_classes=10).to(device="cuda")
evaluate(model , valid_data_loader , metrics_fn=accuracy)


tensor(0.8848, device='cuda:0')